# Cadeia ENSO → chuva → rendimento da soja

**Por que este notebook:** os dois notebooks anteriores olham **um elo de cada vez**:
- `eda_enso_defasagens.ipynb`: ENSO → **chuva** (D-01 a D-08);
- `eda_enso_rendimento.ipynb`: ENSO → **rendimento** (D-09 e D-10).

Falta juntar as duas peças. **Perguntas:**
1. **Elo 2:** a chuva da safra se relaciona com o rendimento em cada região? Em qual parte da safra (início, meio ou fim)?
2. **Cadeia:** a relação ENSO × rendimento **passa pela chuva**? Se a chuva for levada em conta, o que sobra da relação com o ENSO?
3. **Sul:** a chuva no Sul responde ao ENSO (D-06), mas o rendimento responde pouco (D-10). A hipótese do D-10 é que importa **quando** a chuva falta. Isso aparece nos dados?

**Unidade de análise:** microrregião × safra, safras **2006–2025** (os anos em que existem as duas coisas: chuva das estações e rendimento da PAM). Cada microrregião é comparada **com ela mesma** (anomalias), e os resultados são resumidos por região.

Esta é uma **análise de dados**, não um modelo de previsão. Decisões em `docs/decisoes.md` (D-11 e D-12).

**Estrutura:**
1. desenho da análise (fixado antes da execução);
2. dados: chuva, rendimento e ENSO por microrregião e safra;
3. elo 1 (ENSO → chuva) nesta amostra;
4. elo 2 (chuva → rendimento), por subperíodo;
5. a cadeia: quanto da relação ENSO × rendimento passa pela chuva;
6. o caso do Sul: safra a safra;
7. sensibilidade;
8. conclusões.

## 1. Desenho da análise (fixado antes da execução)

| Item | Escolha | Por quê |
|---|---|---|
| ENSO | **ONI médio de set(t−1) a abr(t)**, sendo t o ano da colheita | é a janela da safra e a mesma usada na classificação de fases (D-06). Não usa a janela de máx\|ρ\| do D-10, para não reaproveitar uma escolha feita olhando os dados |
| Chuva | **anomalia % do total** da microrregião, na safra (set–abr, ≥ 7 dos 8 meses) e em três subperíodos: **início** (set–nov), **meio** (dez–fev) e **fim** (mar–abr), com todos os meses válidos | mesma chuva do D-05 e do D-06 (estações dentro da microrregião, completude de 80%) |
| Rendimento | **anomalia % em relação à tendência** LOWESS robusta de 25 anos, por microrregião (D-09) | a tendência é estimada com a série inteira (1974–2025), mais estável que só 2006–2025 |
| Amostra | microrregiões com soja, chuva e rendimento em **≥ 10 safras** | |
| Regiões | **bioma** (D-01) e, como sensibilidade, as UFs **MT, GO, PR, RS** e **MATOPIBA (MA + PI)**, que tiveram os resultados mais claros no D-10 | |
| Correlação | **Spearman** sobre todos os pares microrregião × safra da região (painel) | as anomalias já são relativas a cada microrregião, então podem ser somadas no painel |
| p-valor | **permutação das safras** (5.000): os anos da variável x são embaralhados em bloco, para todas as microrregiões juntas | as microrregiões de um mesmo ano não são independentes (a mesma seca atinge várias); embaralhar anos inteiros preserva isso |
| Intervalo | **bootstrap das safras** (2.000), pelo mesmo motivo | |
| Testes múltiplos | **FDR (Benjamini–Hochberg)** dentro de cada tabela | |
| Níveis | **confirmado:** p corrigido < 0,05; **indício:** p < 0,05 sem correção; **não confirmado:** p ≥ 0,05 | |

**A cadeia (seção 5)** compara a correlação **total** ρ(ONI, rendimento) com a correlação **parcial** ρ(ONI, rendimento | chuva da safra). Se a relação passa pela chuva, a parcial fica perto de zero. A **fração que passa pela chuva** = 1 − parcial ÷ total, com IC por bootstrap das safras. Ela **só é interpretada** onde os dois elos (ENSO → chuva e chuva → rendimento) e a relação total têm p < 0,05; nas outras regiões, a cadeia é dada como "não estabelecida".

**Hipótese do Sul (seção 6), fixada antes:** no Pampa, RS e PR, a chuva do **meio da safra** (dez–fev, floração e enchimento de grãos) se relaciona mais com o rendimento do que a chuva da safra inteira. Teste: IC por bootstrap da diferença ρ(meio) − ρ(total).

**Limitações conhecidas antes de rodar:**
- são só **20 safras**; o painel tem muitos pares, mas a informação independente vem dos anos;
- os subperíodos são de calendário; a fenologia real varia por região e por ano de plantio;
- a correlação de Spearman mede relações **monótonas**: chuva demais também prejudica, e isso aparece na seção 4 como curva, mas não no ρ.

In [ ]:
import warnings
from pathlib import Path

import geobr
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.multitest import multipletests

from soja_clima.geo import mapa_estacoes_cd_micro

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

PROCESSED = Path("../../data/processed")
RAW = Path("../../data/raw")

P = dict(
    meses_safra=[9, 10, 11, 12, 1, 2, 3, 4],
    subperiodos={"início (set–nov)": [9, 10, 11], "meio (dez–fev)": [12, 1, 2], "fim (mar–abr)": [3, 4]},
    completude_mensal=0.80,
    safra_ini=2006, safra_fim=2025,
    min_safras_micro=10,
    # rendimento (iguais ao D-09)
    area_min_ha=100, rendimento_max=6000, min_anos_tendencia=15, janela_tendencia_anos=25,
    n_perm=5000, n_boot=2000, seed=42, alpha=0.05,
)

## 2. Dados
### 2a. Chuva por microrregião e safra
Mesma construção do `eda_enso_defasagens.ipynb` (seção 2): soma mensal por estação (aceita com ≥ 80% dos dias), só estações **dentro do polígono** da microrregião, média entre as estações. A anomalia é o total do período dividido pela média da microrregião naquele período, menos 1, em %.

In [ ]:
clima = pd.read_parquet(PROCESSED / "clima_diario.parquet", columns=["codigo_estacao", "data", "prec_total"])
clima["data"] = pd.to_datetime(clima["data"])
mapa = mapa_estacoes_cd_micro(PROCESSED / "clima_diario.parquet").dropna(subset=["CD_MICRO"])
mapa = mapa[mapa["origem"] == "dentro_do_poligono"]

clima["mes_ref"] = clima["data"].dt.to_period("M")
mensal = (clima.groupby(["codigo_estacao", "mes_ref"])
               .agg(chuva=("prec_total", "sum"), dias=("prec_total", "count")).reset_index())
mensal = mensal[mensal["dias"] >= P["completude_mensal"] * mensal["mes_ref"].dt.days_in_month]
chuva_micro = (mensal.merge(mapa[["codigo_estacao", "CD_MICRO"]], on="codigo_estacao")
                     .groupby(["CD_MICRO", "mes_ref"])["chuva"].mean().reset_index())
chuva_micro["mes"] = chuva_micro["mes_ref"].dt.month
chuva_micro["safra"] = np.where(chuva_micro["mes"] >= 9, chuva_micro["mes_ref"].dt.year + 1, chuva_micro["mes_ref"].dt.year)


def anomalia_chuva(meses: list[int], min_meses: int) -> pd.DataFrame:
    cm = chuva_micro[chuva_micro["mes"].isin(meses)]
    tot = cm.groupby(["CD_MICRO", "safra"]).agg(total=("chuva", "sum"), n=("chuva", "size")).reset_index()
    tot = tot[tot["n"] >= min_meses]
    tot = tot[tot.groupby("CD_MICRO")["safra"].transform("size") >= P["min_safras_micro"]]
    tot["anom"] = 100 * (tot["total"] / tot.groupby("CD_MICRO")["total"].transform("mean") - 1)
    return tot.pivot(index="safra", columns="CD_MICRO", values="anom")


CHUVA = {"safra (set–abr)": anomalia_chuva(P["meses_safra"], 7)}
for nome, meses in P["subperiodos"].items():
    CHUVA[nome] = anomalia_chuva(meses, len(meses))
for k, v in CHUVA.items():
    print(f"chuva {k:18s}: {v.shape[1]} microrregiões | safras {v.index.min()}–{v.index.max()}")

### 2b. Rendimento por microrregião e safra
Mesma construção do `eda_enso_rendimento.ipynb` (D-09): PAM por município (Tabela 1612), agregada por microrregião pelo centroide na malha de 2020, filtros da base de modelagem e tendência **LOWESS robusta de 25 anos** por microrregião, ajustada na série inteira. Usa os CSVs já baixados em `data/raw/` (o notebook de rendimento os baixa, se faltarem).

In [ ]:
ARQ_MUN = RAW / "pam_soja_municipio_1974_2025.csv"
if not ARQ_MUN.exists():
    raise FileNotFoundError(f"{ARQ_MUN} não existe: execute antes notebooks/eda/eda_enso_rendimento.ipynb (ele baixa a PAM).")
mun = pd.read_csv(ARQ_MUN, dtype={"codigo": str})

municipios = geobr.read_municipality(year=2020)
micro = geobr.read_micro_region(year=2020)
micro["CD_MICRO"] = micro["code_micro"].astype(int).astype(str)
cent = municipios.to_crs(5880)
cent["geometry"] = cent.centroid
cent = cent.to_crs(micro.crs)
j = gpd.sjoin(cent[["code_muni", "geometry"]], micro[["CD_MICRO", "geometry"]], predicate="within")
muni_micro = pd.DataFrame({"codigo": j["code_muni"].astype(int).astype(str), "CD_MICRO": j["CD_MICRO"]})

micro_ano = (mun.merge(muni_micro, on="codigo").groupby(["CD_MICRO", "ano"])[["producao_t", "area_colhida_ha"]]
                .sum(min_count=1).reset_index().dropna())
r = micro_ano[micro_ano["area_colhida_ha"] >= P["area_min_ha"]].copy()
r["rend"] = r["producao_t"] * 1000 / r["area_colhida_ha"]
r = r[r["rend"].between(1, P["rendimento_max"])]
r = r[r.groupby("CD_MICRO")["ano"].transform("size") >= P["min_anos_tendencia"]]


def tendencia_lowess(g: pd.DataFrame) -> pd.Series:
    anos, y = g["ano"].to_numpy(float), g["rend"].to_numpy(float)
    frac = min(1.0, P["janela_tendencia_anos"] / (anos.max() - anos.min() + 1))
    return pd.Series(lowess(y, anos, frac=frac, it=3, return_sorted=False), index=g.index)


r["tend"] = r.groupby("CD_MICRO", group_keys=False)[["ano", "rend"]].apply(tendencia_lowess)
r["anom"] = 100 * (r["rend"] / r["tend"] - 1)
REND = r.pivot(index="ano", columns="CD_MICRO", values="anom")
AREA = r.groupby("CD_MICRO")["area_colhida_ha"].mean()
print(f"rendimento: {REND.shape[1]} microrregiões | safras {REND.index.min()}–{REND.index.max()}")

### 2c. ENSO, regiões e amostra final

In [ ]:
pac = pd.read_parquet(PROCESSED / "pacifico_indices.parquet")
pac["safra"] = np.where(pac["mes"] >= 9, pac["ano"] + 1, pac["ano"])
ONI = pac[pac["mes"].isin(P["meses_safra"])].groupby("safra")["oni"].agg(["mean", "size"])
ONI = ONI.loc[ONI["size"] == 8, "mean"]

# Bioma por maior área (D-01, Pantanal -> Cerrado) e UF
biomas = geobr.read_biomes(year=2019)
biomas = biomas[~biomas["name_biome"].str.contains("Costeiro", case=False, na=False)]
m_ = micro[["CD_MICRO", "geometry"]].to_crs(5880)
b_ = biomas[["name_biome", "geometry"]].to_crs(5880)
m_["geometry"], b_["geometry"] = m_.geometry.make_valid(), b_.geometry.make_valid()
inter = gpd.overlay(m_, b_, how="intersection", keep_geom_type=True)
inter["area"] = inter.area
BIOMA = (inter.sort_values("area").drop_duplicates("CD_MICRO", keep="last").set_index("CD_MICRO")["name_biome"]
              .replace({"Pantanal": "Cerrado"}))
UF = micro.set_index("CD_MICRO")["abbrev_state"]

SAFRAS = np.arange(P["safra_ini"], P["safra_fim"] + 1)
REND = REND.reindex(SAFRAS)
CHUVA = {k: v.reindex(SAFRAS) for k, v in CHUVA.items()}
X_ONI = ONI.reindex(SAFRAS)

# microrregiões da amostra: chuva da safra e rendimento juntos em >= 10 safras
comum = REND.columns.intersection(CHUVA["safra (set–abr)"].columns)
pares = (REND[comum].notna() & CHUVA["safra (set–abr)"][comum].notna()).sum()
MICROS = pares[pares >= P["min_safras_micro"]].index

REGIOES = {b: MICROS[BIOMA.reindex(MICROS).to_numpy() == b] for b in ["Amazônia", "Caatinga", "Cerrado", "Mata Atlântica", "Pampa"]}
for uf in ["MT", "GO", "PR", "RS"]:
    REGIOES[f"UF {uf}"] = MICROS[UF.reindex(MICROS).to_numpy() == uf]
REGIOES["MATOPIBA (MA + PI)"] = MICROS[UF.reindex(MICROS).isin(["MA", "PI"]).to_numpy()]

resumo = pd.DataFrame({
    "microrregiões": {k: len(v) for k, v in REGIOES.items()},
    "pares micro × safra": {k: int((REND[v].notna() & CHUVA["safra (set–abr)"][v].notna()).sum().sum()) for k, v in REGIOES.items()},
    "% da área de soja da região na amostra": {
        k: round(100 * AREA.reindex(v).sum() / AREA[(BIOMA.reindex(AREA.index) == k).to_numpy() if not k.startswith(("UF", "MATOPIBA"))
                                                  else (UF.reindex(AREA.index).isin([k[3:]] if k.startswith("UF") else ["MA", "PI"])).to_numpy()].sum(), 1)
        for k, v in REGIOES.items()},
})
print(f"{len(MICROS)} microrregiões na amostra | ONI da safra {X_ONI.min():+.2f} a {X_ONI.max():+.2f}")
resumo

**Leitura:** as regiões com poucas microrregiões (Caatinga, Amazônia, MATOPIBA) têm resultados mais frágeis. A coluna de área mostra quanto da soja da região está coberta por estações; uma cobertura baixa quer dizer que a amostra pode não representar a região inteira.

### Funções estatísticas
- `rho_painel`: Spearman sobre todos os pares microrregião × safra válidos.
- `perm_safras`: p bilateral pela permutação das safras da variável x (em bloco, todas as microrregiões juntas).
- `boot_safras`: reamostra safras com reposição e recalcula a estatística.
- `parcial`: correlação parcial de Spearman (postos) de y e x, controlando z.

In [ ]:
RNG = np.random.default_rng(P["seed"])
PERMS = np.array([RNG.permutation(len(SAFRAS)) for _ in range(P["n_perm"])])
BOOTS = RNG.integers(0, len(SAFRAS), size=(P["n_boot"], len(SAFRAS)))


def como_matriz(x, micros) -> np.ndarray:
    # ONI (série por safra) vira matriz safra × micro; anomalias já são matrizes
    if isinstance(x, pd.Series):
        return np.repeat(x.to_numpy(float)[:, None], len(micros), axis=1)
    return x.reindex(columns=micros).to_numpy(float)


def _rho(a: np.ndarray, b: np.ndarray) -> float:
    ok = ~(np.isnan(a) | np.isnan(b))
    if ok.sum() < 10:
        return np.nan
    ra, rb = rankdata(a[ok]), rankdata(b[ok])
    return float(np.corrcoef(ra, rb)[0, 1])


def _parcial(y, x, z) -> float:
    ok = ~(np.isnan(y) | np.isnan(x) | np.isnan(z))
    if ok.sum() < 10:
        return np.nan
    c = np.corrcoef(np.vstack([rankdata(y[ok]), rankdata(x[ok]), rankdata(z[ok])]))
    ryx, ryz, rxz = c[0, 1], c[0, 2], c[1, 2]
    return float((ryx - ryz * rxz) / np.sqrt((1 - ryz ** 2) * (1 - rxz ** 2)))


def rho_painel(X, Y) -> float:
    return _rho(X.ravel(), Y.ravel())


def perm_safras(X, Y) -> float:
    obs = rho_painel(X, Y)
    nul = np.array([_rho(X[p].ravel(), Y.ravel()) for p in PERMS])
    return float((1 + np.sum(np.abs(nul) >= abs(obs))) / (1 + len(nul)))


def boot_safras(func, *mats) -> np.ndarray:
    return np.array([func(*[m[b] for m in mats]) for b in BOOTS])


def nivel(p_bh: float, p: float) -> str:
    return "confirmado" if p_bh < P["alpha"] else ("indício" if p < P["alpha"] else "não confirmado")


def tabela_elo(pares: dict) -> pd.DataFrame:
    # pares: {(região, variável): (X, Y)} -> ρ, IC, p e nível com FDR na tabela inteira
    lin = []
    for (reg, var), (X, Y) in pares.items():
        rho = rho_painel(X, Y)
        bs = boot_safras(lambda a, b: rho_painel(a, b), X, Y)
        lin.append(dict(região=reg, variável=var, n=int((~np.isnan(X) & ~np.isnan(Y)).sum()), ρ=rho,
                        IC95=f"[{np.nanpercentile(bs, 2.5):+.2f}; {np.nanpercentile(bs, 97.5):+.2f}]", p=perm_safras(X, Y)))
    t = pd.DataFrame(lin)
    t["p (FDR)"] = multipletests(t["p"], method="fdr_bh")[1]
    t["nível"] = [nivel(a, b) for a, b in zip(t["p (FDR)"], t["p"])]
    return t.round({"ρ": 3, "p": 4, "p (FDR)": 4})

## 3. Elo 1: ENSO → chuva (nesta amostra)
Já documentado no D-03 a D-06, com a chuva mensal. Aqui é recalculado **na mesma amostra e na mesma escala** (safra e subperíodos) usada nos outros elos, para que a cadeia seja comparável. Sinal esperado: ONI positivo (El Niño) → **mais** chuva no Sul e **menos** no Norte.

In [ ]:
VARS_CHUVA = list(CHUVA)
elo1 = tabela_elo({(reg, var): (como_matriz(X_ONI, m), como_matriz(CHUVA[var], m))
                   for reg, m in REGIOES.items() for var in VARS_CHUVA})
elo1.pivot(index="região", columns="variável", values="ρ")[VARS_CHUVA].reindex(list(REGIOES)).round(2)

In [ ]:
elo1.pivot(index="região", columns="variável", values="nível")[VARS_CHUVA].reindex(list(REGIOES))

## 4. Elo 2: chuva → rendimento
Para cada região e período da safra: Spearman entre a anomalia de chuva e a anomalia de rendimento, no painel microrregião × safra.

In [ ]:
elo2 = tabela_elo({(reg, var): (como_matriz(CHUVA[var], m), como_matriz(REND, m))
                   for reg, m in REGIOES.items() for var in VARS_CHUVA})
elo2.pivot(index="região", columns="variável", values="ρ")[VARS_CHUVA].reindex(list(REGIOES)).round(2)

In [ ]:
elo2.pivot(index="região", columns="variável", values="nível")[VARS_CHUVA].reindex(list(REGIOES))

### A forma da relação
O ρ mede só relações monótonas. O gráfico mostra a **anomalia média de rendimento por faixa de anomalia de chuva da safra** (decis do painel de cada bioma): uma curva que sobe e depois cai indica que chuva demais também prejudica.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(21, 3.6), sharey=True, layout="constrained")
for ax, b in zip(axes, ["Amazônia", "Caatinga", "Cerrado", "Mata Atlântica", "Pampa"]):
    m = REGIOES[b]
    x, y = como_matriz(CHUVA["safra (set–abr)"], m).ravel(), como_matriz(REND, m).ravel()
    ok = ~(np.isnan(x) | np.isnan(y))
    x, y = x[ok], y[ok]
    if len(x) < 30:
        ax.set_title(f"{b} (n={len(x)}, poucos dados)")
        ax.axis("off")
        continue
    faixa = pd.qcut(x, 10, duplicates="drop")
    g = pd.DataFrame({"x": x, "y": y, "f": faixa}).groupby("f", observed=True).agg(x=("x", "median"), y=("y", "mean"),
                                                                                    e=("y", "sem"))
    ax.scatter(x, y, s=4, alpha=0.15, color="grey")
    ax.errorbar(g["x"], g["y"], yerr=1.96 * g["e"], color="#1b9e77", marker="o", lw=1.8, capsize=2)
    ax.axhline(0, color="k", lw=0.6)
    ax.axvline(0, color="k", lw=0.6)
    ax.set_xlim(np.percentile(x, [1, 99]))
    ax.set_ylim(-60, 40)
    ax.set_title(f"{b} (n={len(x)})")
    ax.set_xlabel("anomalia de chuva da safra (%)")
axes[0].set_ylabel("anomalia de rendimento (%)")
fig.suptitle("Rendimento médio por decil de chuva da safra (barras: IC 95% da média)")
plt.show()

## 5. A cadeia: quanto da relação ENSO × rendimento passa pela chuva
Para cada região:
- **ρ total** = ρ(ONI, rendimento);
- **ρ parcial** = ρ(ONI, rendimento | chuva da safra);
- **fração pela chuva** = 1 − parcial ÷ total, com IC por bootstrap das safras.

A fração só é interpretada onde **os três** (ENSO → chuva, chuva → rendimento e ENSO → rendimento) têm p < 0,05 (regra fixada na seção 1). O mesmo cálculo é feito controlando a chuva dos três subperíodos juntos (correlação parcial com três variáveis de controle), porque a chuva da safra inteira pode esconder o momento em que ela faltou.

In [ ]:
def parcial_multi(y, x, Z) -> float:
    # Spearman parcial de y e x controlando várias colunas Z (resíduos dos postos por mínimos quadrados)
    cols = [y, x] + list(Z)
    ok = ~np.any(np.isnan(np.vstack(cols)), axis=0)
    if ok.sum() < 15:
        return np.nan
    R = np.column_stack([rankdata(c[ok]) for c in cols])
    A = np.column_stack([np.ones(ok.sum()), R[:, 2:]])
    res = R[:, :2] - A @ np.linalg.lstsq(A, R[:, :2], rcond=None)[0]
    return float(np.corrcoef(res[:, 0], res[:, 1])[0, 1])


def frac(total, parc):
    return 1 - parc / total if abs(total) > 1e-9 else np.nan


ROTULO_ELO1 = elo1.set_index(["região", "variável"])["p"]
ROTULO_ELO2 = elo2.set_index(["região", "variável"])["p"]
SUB = [v for v in VARS_CHUVA if v != "safra (set–abr)"]
lin = []
for reg, m in REGIOES.items():
    Xo, Y, C = como_matriz(X_ONI, m), como_matriz(REND, m), como_matriz(CHUVA["safra (set–abr)"], m)
    Cs = [como_matriz(CHUVA[v], m) for v in SUB]
    # mesma amostra para total e parcial: pares com chuva da safra
    Yc = np.where(np.isnan(C), np.nan, Y)
    tot = rho_painel(Xo, Yc)
    p_tot = perm_safras(Xo, Yc)
    par = _parcial(Yc.ravel(), Xo.ravel(), C.ravel())
    bs = boot_safras(lambda a, b, c: frac(rho_painel(a, np.where(np.isnan(c), np.nan, b)),
                                          _parcial(np.where(np.isnan(c), np.nan, b).ravel(), a.ravel(), c.ravel())), Xo, Y, C)
    # versão com os três subperíodos (amostra com os três disponíveis)
    ok3 = ~np.any(np.isnan(np.stack(Cs)), axis=0)
    Y3 = np.where(ok3, Y, np.nan)
    tot3 = rho_painel(Xo, Y3)
    par3 = parcial_multi(Y3.ravel(), Xo.ravel(), [c.ravel() for c in Cs])
    cadeia = (p_tot < P["alpha"]) and (ROTULO_ELO1[(reg, "safra (set–abr)")] < P["alpha"]) and (ROTULO_ELO2[(reg, "safra (set–abr)")] < P["alpha"])
    lin.append({"região": reg, "ρ total": tot, "p total": p_tot, "ρ parcial (chuva safra)": par,
                "fração pela chuva": frac(tot, par),
                "IC95 fração": f"[{np.nanpercentile(bs, 2.5):+.2f}; {np.nanpercentile(bs, 97.5):+.2f}]",
                "ρ total (amostra 3 sub.)": tot3, "ρ parcial (3 subperíodos)": par3, "fração (3 sub.)": frac(tot3, par3),
                "cadeia": "interpretável" if cadeia else "não estabelecida"})
cad = pd.DataFrame(lin).set_index("região")
cad.round(3)

**Como ler:** fração perto de 1 = a relação ENSO × rendimento é quase toda acompanhada pela chuva medida; perto de 0 = o ENSO se relaciona com o rendimento por outro caminho (temperatura, radiação, distribuição da chuva dentro do mês, pragas) ou a chuva das estações não representa bem a microrregião. Valores acima de 1 ou negativos aparecem quando a relação total é fraca e não devem ser interpretados.

### Visualização da cadeia
Para cada bioma, os três ρ: ENSO → chuva, chuva → rendimento e ENSO → rendimento (total e parcial).

In [ ]:
biomas_ord = ["Amazônia", "Caatinga", "Cerrado", "Mata Atlântica", "Pampa", "UF MT", "UF GO", "UF PR", "UF RS", "MATOPIBA (MA + PI)"]
e1 = elo1[elo1["variável"] == "safra (set–abr)"].set_index("região")["ρ"]
e2 = elo2[elo2["variável"] == "safra (set–abr)"].set_index("região")["ρ"]
d = pd.DataFrame({"ENSO → chuva": e1, "chuva → rendimento": e2, "ENSO → rendimento (total)": cad["ρ total"],
                  "ENSO → rendimento (parcial | chuva)": cad["ρ parcial (chuva safra)"]}).reindex(biomas_ord)
ax = d.plot.bar(figsize=(14, 4.2), width=0.8, color=["#7570b3", "#1b9e77", "#d95f02", "#fdb863"])
ax.axhline(0, color="k", lw=0.7)
ax.set_ylabel("ρ de Spearman (painel)")
ax.set_title("Cadeia ENSO → chuva → rendimento, safras 2006–2025")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 6. O caso do Sul
**Hipótese (fixada na seção 1):** no Pampa, RS e PR, a chuva do **meio da safra** (dez–fev) se relaciona mais com o rendimento do que a chuva da safra inteira. Teste: diferença ρ(meio) − ρ(safra), com IC por bootstrap das safras.

In [ ]:
lin = []
for reg in ["Pampa", "UF RS", "UF PR", "Cerrado", "UF MT"]:
    m = REGIOES[reg]
    Y = como_matriz(REND, m)
    Cs, Cm = como_matriz(CHUVA["safra (set–abr)"], m), como_matriz(CHUVA["meio (dez–fev)"], m)
    ok = ~(np.isnan(Cs) | np.isnan(Cm))
    Ys = np.where(ok, Y, np.nan)
    dif = rho_painel(Cm, Ys) - rho_painel(Cs, Ys)
    bs = boot_safras(lambda a, b, c: rho_painel(b, c) - rho_painel(a, c), Cs, Cm, Ys)
    lin.append({"região": reg, "ρ safra": rho_painel(Cs, Ys), "ρ meio (dez–fev)": rho_painel(Cm, Ys), "diferença": dif,
                "IC95": f"[{np.nanpercentile(bs, 2.5):+.2f}; {np.nanpercentile(bs, 97.5):+.2f}]",
                "meio > safra": bool(np.nanpercentile(bs, 2.5) > 0)})
pd.DataFrame(lin).set_index("região").round(3)

### Safra a safra no Pampa
Média das microrregiões (ponderada pela área de soja) de cada safra, com a fase do ENSO (critério do D-06). Serve para ver se as quebras de safra em anos **neutros** (D-10) tiveram falta de chuva em algum subperíodo.

In [ ]:
def fase_mensal(oni: pd.Series, limiar: float = 0.5, n_min: int = 5) -> pd.Series:
    out = pd.Series("neutro", index=oni.index)
    for nome, cond in (("El Niño", oni >= limiar), ("La Niña", oni <= -limiar)):
        grupo = (cond != cond.shift()).cumsum()
        out[cond & (cond.groupby(grupo).transform("size") >= n_min)] = nome
    return out


pac_m = pac.assign(mes_ref=pd.to_datetime(dict(year=pac["ano"], month=pac["mes"], day=1)).dt.to_period("M")).set_index("mes_ref").sort_index()
fm = fase_mensal(pac_m["oni"].dropna())
jan = fm[np.isin(fm.index.month, P["meses_safra"])]
saf = pd.Series(np.where(jan.index.month >= 9, jan.index.year + 1, jan.index.year), index=jan.index)
FASE = jan.groupby(saf).apply(lambda s: "El Niño" if (s == "El Niño").sum() >= 4 else ("La Niña" if (s == "La Niña").sum() >= 4 else "neutro")).reindex(SAFRAS)


def media_regional(df: pd.DataFrame, micros) -> pd.Series:
    w = AREA.reindex(micros)
    v = df.reindex(columns=micros)
    return (v * w).sum(axis=1, min_count=1) / v.notna().mul(w).sum(axis=1).replace(0, np.nan)


def tabela_safras(reg: str) -> pd.DataFrame:
    m = REGIOES[reg]
    t = pd.DataFrame({"fase": FASE, "ONI": X_ONI.round(2), "rendimento (%)": media_regional(REND, m)})
    for v in VARS_CHUVA:
        t[f"chuva {v} (%)"] = media_regional(CHUVA[v], m)
    return t.round(1)


tab_pampa = tabela_safras("Pampa")
tab_pampa.style.background_gradient(cmap="RdBu", vmin=-40, vmax=40, subset=[c for c in tab_pampa.columns if "(%)" in c]).format(precision=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), layout="constrained")
cores = {"El Niño": "#d7301f", "La Niña": "#2b8cbe", "neutro": "#636363"}
for ax, reg in zip(axes, ["Pampa", "Cerrado"]):
    t = tabela_safras(reg)
    for var, mk in [("chuva safra (set–abr) (%)", "o"), ("chuva meio (dez–fev) (%)", "^")]:
        ax.scatter(t[var], t["rendimento (%)"], c=t["fase"].map(cores), marker=mk, s=45, alpha=0.85,
                   label=var.replace(" (%)", ""))
    for s_, l in t.iterrows():
        ax.annotate(str(s_)[2:], (l["chuva meio (dez–fev) (%)"], l["rendimento (%)"]), fontsize=7, xytext=(3, 2), textcoords="offset points")
    ax.axhline(0, color="k", lw=0.6)
    ax.axvline(0, color="k", lw=0.6)
    ax.set_title(f"{reg}: média regional por safra (cor = fase do ENSO)")
    ax.set_xlabel("anomalia de chuva (%)  ·  círculo = safra inteira, triângulo = dez–fev (rótulo = safra)")
    ax.set_ylabel("anomalia de rendimento (%)")
plt.show()

## 7. Sensibilidade
1. **Média regional** em vez do painel: correlação entre as séries regionais (20 safras, média ponderada pela área), como no D-10. Mede a mesma coisa num nível agregado.
2. **Sem as microrregiões com só uma estação**, que podem medir mal a chuva da microrregião.

In [ ]:
n_est = mapa.groupby("CD_MICRO")["codigo_estacao"].nunique()
lin = []
for reg, m in REGIOES.items():
    o, ch, re_ = X_ONI, media_regional(CHUVA["safra (set–abr)"], m), media_regional(REND, m)
    d = pd.DataFrame({"o": o, "c": ch, "r": re_}).dropna()
    m2 = m[n_est.reindex(m).fillna(0).to_numpy() >= 2]
    lin.append({"região": reg, "safras": len(d),
                "regional: ENSO→chuva": d["o"].corr(d["c"], method="spearman"),
                "regional: chuva→rend.": d["c"].corr(d["r"], method="spearman"),
                "regional: ENSO→rend.": d["o"].corr(d["r"], method="spearman"),
                "micros com ≥2 estações": len(m2),
                "painel ≥2 est.: chuva→rend.": rho_painel(como_matriz(CHUVA["safra (set–abr)"], m2), como_matriz(REND, m2)) if len(m2) else np.nan,
                "painel (todas): chuva→rend.": rho_painel(como_matriz(CHUVA["safra (set–abr)"], m), como_matriz(REND, m))})
pd.DataFrame(lin).set_index("região").round(2)

### 7c. A cadeia no nível regional (acrescentada depois da primeira execução)
**Por que:** na primeira execução, a relação chuva → rendimento foi bem mais forte nas **médias regionais** (por exemplo, MATOPIBA: ρ = 0,74) do que no **painel** de microrregiões (ρ = 0,14). Isso sugere que a chuva de **uma ou duas estações** mede mal a chuva da microrregião inteira, e esse erro de medida faz a correlação parcial ficar perto da total (a fração "pela chuva" fica subestimada). A média regional reduz o erro, então a cadeia é refeita nesse nível. Como foi decidida depois de ver os dados e tem só cerca de 20 safras, é **exploratória**.

In [ ]:
lin = []
for reg, m in REGIOES.items():
    d = pd.DataFrame({"o": X_ONI, "c": media_regional(CHUVA["safra (set–abr)"], m),
                      "cm": media_regional(CHUVA["meio (dez–fev)"], m), "r": media_regional(REND, m)}).dropna()
    if len(d) < 12:
        continue
    o, c, cm, y = (d[k].to_numpy() for k in ("o", "c", "cm", "r"))
    tot = d["o"].corr(d["r"], method="spearman")
    par, par_m = _parcial(y, o, c), _parcial(y, o, cm)
    idx = RNG.integers(0, len(d), size=(P["n_boot"], len(d)))
    bs = np.array([frac(_rho(o[b], y[b]), _parcial(y[b], o[b], c[b])) for b in idx])
    lin.append({"região": reg, "safras": len(d), "ρ total": tot, "ρ parcial | chuva safra": par,
                "fração pela chuva safra": frac(tot, par),
                "IC95": f"[{np.nanpercentile(bs, 2.5):+.2f}; {np.nanpercentile(bs, 97.5):+.2f}]",
                "ρ parcial | chuva dez–fev": par_m, "fração pela chuva dez–fev": frac(tot, par_m)})
pd.DataFrame(lin).set_index("região").round(2)

## 8. Conclusões

**Amostra:** 153 microrregiões, safras 2006–2025. A cobertura por estações é boa no Cerrado (95% da área de soja), no Pampa (99%), no RS, no MT e em GO (85–93%), e mais fraca na Amazônia (59%), na Mata Atlântica (56%) e no PR (45%). A **Caatinga tem uma única microrregião** e não é interpretada.

**Elo 1 (ENSO → chuva), nesta escala:**
- **Sul:** relação forte e **concentrada no início da safra** (set–nov: Pampa ρ = 0,64; RS 0,63). Ela é mais fraca no meio (dez–fev: 0,37; indício) e nula no fim.
- **Centro-Norte:** com o ONI médio da safra, a relação com a chuva é fraca. Há só indícios no meio da safra (GO −0,30; Amazônia −0,25; Cerrado −0,16). Isso é coerente com o D-05: no Cerrado, a relação muda de sinal dentro da safra e aparece com defasagem de 7 meses, que o ONI médio da safra não capta.

**Elo 2 (chuva → rendimento):**
- **A chuva que importa é a do meio da safra (dez–fev)**, na floração e no enchimento de grãos: a relação é confirmada no Pampa (ρ = 0,45), no RS (0,47), no PR (0,31), na Mata Atlântica (0,29) e no Cerrado (0,16). A chuva do início e do fim da safra não se relaciona com o rendimento.
- A relação **não é linear**. No Pampa, o rendimento é máximo com chuva cerca de 10% acima da média e **não melhora (ou piora) com chuva bem acima da média**.
- No MT e no MATOPIBA, a chuva das estações quase não se relaciona com o rendimento no painel (ρ ≈ 0,05–0,14). Nas médias regionais, a relação é mais forte (MT 0,32; MATOPIBA 0,74), sinal de que **uma ou duas estações medem mal a chuva da microrregião**.

**A cadeia (seção 5 e 7c):**
- Pela regra fixada antes da execução, a cadeia **não fica estabelecida em nenhuma região**: em nenhuma delas os três elos são significativos ao mesmo tempo.
- **Centro-Norte (Cerrado, MT, GO, MATOPIBA):** o ENSO se relaciona claramente com o rendimento (painel: MT −0,44, MATOPIBA −0,46, Cerrado −0,32; regional: −0,52 a −0,76). Mas **quase nada dessa relação é acompanhado pela chuva medida**: a fração pela chuva fica perto de zero no painel (Cerrado: IC de −0,03 a +0,08; MT: de −0,05 a +0,06) e também no nível regional, onde o erro de medida é menor (MT 0,04; MATOPIBA 0,01; Cerrado −0,02). Controlar só a chuva de dez–fev dá o mesmo resultado.

  **Leitura:** no Centro-Norte, o El Niño se relaciona com rendimento menor **por outro caminho além do total de chuva**. Os candidatos são o **calor** (o D-08 mostrou El Niño → Tmax maior no Norte e no Centro-Oeste, um sinal robusto), a **distribuição** da chuva dentro do mês (veranicos curtos) e a radiação. Isso não foi testado aqui.
- **Sul:** a relação total ENSO × rendimento é quase nula (ρ ≈ 0), então não há o que decompor. Depois de considerar a chuva, porém, a relação fica **negativa** (Pampa: −0,22 no painel, −0,19 no regional): com a mesma chuva, as safras de El Niño rendem um pouco menos. É coerente com o excesso de chuva e de nebulosidade, mas é **exploratório**.

**O caso do Sul, explicado (seção 6):**
1. O ENSO age mais sobre a chuva do **início** da safra, que **não** se relaciona com o rendimento.
2. A chuva que importa (**dez–fev**) responde menos ao ENSO.
3. O excesso de chuva do El Niño **não aumenta** o rendimento (a curva se achata).

A **hipótese fixada antes foi confirmada:** no Pampa, RS e PR, a chuva de dez–fev se relaciona mais com o rendimento do que a da safra inteira (diferença no Pampa: +0,25, IC de +0,07 a +0,42).

**Safra a safra:** todas as grandes quebras do Pampa tiveram dez–fev seco, **qualquer que fosse a fase**:
- La Niña: 2012 (−48%), 2022 (−51%), 2023 (−41%);
- El Niño fraco: 2020 (−35%, dez–fev −34%);
- neutra: 2025 (−32%, dez–fev −42%).

E a La Niña de 2021 teve a safra 30% mais seca, mas dez–fev só 11% abaixo da média, e rendeu **+25%**.

**Cautelas:**
- 20 safras: a informação independente vem dos anos, e os IC são largos no nível regional.
- A decomposição por correlação parcial é **descritiva**: mostra o que acompanha o quê, não prova o mecanismo.
- O erro de medida da chuva (poucas estações por microrregião) puxa a fração "pela chuva" para baixo. A checagem regional (7c) reduz esse problema, e o resultado do Centro-Norte se manteve.
- Os subperíodos são de calendário; a fenologia real varia por região e por data de plantio.

**Para as próximas etapas:**
- no **Sul**, a variável climática relevante é a **chuva de dez–fev**, e não o índice ENSO;
- no **Centro-Norte**, o ENSO carrega informação que a chuva das estações não carrega. O próximo teste natural é a **temperatura** como caminho (a mesma cadeia, com Tmax no lugar da chuva).